In [1]:
import sys
sys.path.append("..")
from pathlib import Path
import pandas as pd
from src.pse_client import fetch_load

wczoraj = (pd.Timestamp.today() - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
df = fetch_load("2024-06-14", wczoraj)  # ok. 860 dni, potrwa kilka–kilkanaście minut

Path("../data/raw").mkdir(parents=True, exist_ok=True)
df.to_parquet("../data/raw/kse_load.parquet", index=False)
print(len(df), "wierszy,", df["business_date"].nunique(), "dni")

82752 wierszy, 862 dni


In [2]:
import pandas as pd

df = pd.read_parquet("../data/raw/kse_load.parquet")
print("Wiersze:", len(df))
print("Zakres:", df["business_date"].min(), "->", df["business_date"].max())

# 1) Duplikaty: ten sam kwadrans (dtime_utc) nie powinien występować dwa razy
print("Duplikaty dtime_utc:", df["dtime_utc"].duplicated().sum())

# 2) Brakujące dni: porównujemy z pełnym kalendarzem
dni = pd.to_datetime(df["business_date"]).dt.date
wszystkie = pd.date_range(dni.min(), dni.max(), freq="D").date
brak = sorted(set(wszystkie) - set(dni))
print("Brakujące dni:", brak)

# 3) Ile wierszy na dobę (normalnie 96)
na_dobe = df.groupby("business_date").size()
print(na_dobe.value_counts())
print("Dni inne niż 96:")
print(na_dobe[na_dobe != 96])

# 4) Puste wartości
print(df[["load_actual", "load_fcst"]].isna().sum())

Wiersze: 82752
Zakres: 2024-06-14 -> 2026-10-23
Duplikaty dtime_utc: 0
Brakujące dni: []
96     858
100      2
92       2
Name: count, dtype: int64
Dni inne niż 96:
business_date
2024-10-27    100
2025-03-30     92
2025-10-26    100
2026-03-29     92
dtype: int64
load_actual    0
load_fcst      0
dtype: int64
